In [ ]:
#!/usr/bin/env python3
"""
ACNE04 consequence experiments: does any of the audit actually change the
numbers people report?

The audit established facts about the dataset. Facts alone invite the
reviewer's question "so what?". These three experiments answer it.

  ARM A   random split, duplicates kept, grade head
          - what the literature effectively does
  ARM B   subject-disjoint split, duplicates removed, grade head
          - the clean protocol
          A -> B is the INFLATION attributable to protocol defects.

  ARM C   subject-disjoint split, duplicates removed, COUNT head,
          grade obtained by thresholding the predicted count
          - if C matches or beats B, the grading head learns nothing the
            counting head does not already determine, which is the central
            claim of the paper demonstrated experimentally rather than
            argued from the label definition.

  THRESHOLD SENSITIVITY  derived from ARM C at no extra cost: re-threshold
          the SAME predicted counts under perturbed cut-points. If accuracy
          swings under a one-lesion change to an arbitrary threshold, then
          reported grading accuracy partly measures threshold placement.

Honest expectation: A -> B may be small, because groups.csv contains few
multi-image groups. Report whatever it is. C and the sensitivity sweep are
where the argument is carried.
"""

from __future__ import annotations

import csv
import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

# --- edit ------------------------------------------------------------------
ROOT = "/kaggle/input/datasets/atkiyaanisa/acne04/acne04"
GROUPS_CSV = "fold_audit_out/groups.csv"
DUPLICATES_CSV = "fold_audit_out/duplicate_groups.csv"
OUT = "consequence_out"

BACKBONE = "resnet18"
IMG_SIZE = 224
EPOCHS = 15            # 15 is enough to separate the arms; 20 adds ~25% time
BATCH = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
N_FOLDS = 5
VAL_FRAC = 0.15
SEED = 0

# Arms A and B already ran cleanly; re-running them wastes GPU quota.
# Set ARMS = ["A","B","C"] to redo everything.
ARMS = ["C", "D"]
REFERENCE = {"A": {"qwk": 0.8523, "acc": 0.7625},     # random split, dups kept
             "B": {"qwk": 0.8602, "acc": 0.7639}}     # clean protocol, CORAL head
CLASS_WEIGHTS = False   # plain cross-entropy first; change one thing at a time
# ---------------------------------------------------------------------------

CUTS = (5, 20, 50)
N_GRADES = 4
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def rule(count, cuts=CUTS):
    g = 0
    for c in cuts:
        if count > c:
            g += 1
        else:
            break
    return g


# ---------------------------------------------------------------------------
# data
# ---------------------------------------------------------------------------

USE_IMAGE_CACHE = True
CACHE_DIR = "/kaggle/working/img_cache"
CACHE_SIZE = 256          # shorter side; the model only ever sees 224


def build_image_cache(rec, size=CACHE_SIZE, cache_dir=CACHE_DIR):
    """
    Resize every image once, then train from the cache.

    ACNE04 images are 3112x3456. Decoding ~10-megapixel JPEGs every epoch makes
    training DATA-bound rather than GPU-bound - earlier runs spent two hours
    with the GPU largely idle, waiting on libjpeg. Caching at 256px cuts that
    to minutes. The model only ever sees 224x224 crops, so nothing it can
    observe changes.

    draft() lets libjpeg decode at a reduced scale directly, which is far
    faster than decoding full size and then shrinking.
    """
    from PIL import Image
    d = Path(cache_dir)
    d.mkdir(parents=True, exist_ok=True)
    made = reused = failed = 0
    for n, r in rec.items():
        dst = d / (Path(n).stem + ".jpg")
        if dst.exists():
            reused += 1
        else:
            try:
                with Image.open(r["path"]) as im:
                    im.draft("RGB", (size, size))
                    im = im.convert("RGB")
                    w, h = im.size
                    s = size / min(w, h)
                    if s < 1.0:
                        im = im.resize((max(1, round(w * s)), max(1, round(h * s))),
                                       Image.BILINEAR)
                    im.save(dst, "JPEG", quality=92)
                made += 1
            except Exception as e:
                failed += 1
                print(f"    cache failed for {n}: {e}")
                continue
        r["path"] = dst
    print(f"  image cache                 : {made} built, {reused} reused"
          + (f", {failed} FAILED (using originals)" if failed else "")
          + f"  -> {d}")
    return rec


def find_file(name, hint=None):
    cands = [Path(hint)] if hint else []
    cands += [Path(name), Path("fold_audit_out") / name]
    for c in cands:
        if c.is_file():
            return c
    for base in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if base.is_dir():
            try:
                hits = sorted(base.rglob(name))
            except OSError:
                hits = []
            if hits:
                print(f"  found {name} at {hits[0]}")
                return hits[0]
    return None


def load_records(root: Path):
    imgs = {}
    for p in root.rglob("*"):
        if p.suffix.lower() in IMAGE_EXTS and p.is_file():
            if p.name not in imgs or "Classification" in p.parts:
                imgs[p.name] = p

    rec = {}
    for p in root.rglob("*.txt"):
        for line in p.read_text(errors="replace").splitlines():
            parts = line.replace(",", " ").split()
            name = next((t for t in parts
                         if Path(t).suffix.lower() in IMAGE_EXTS), None)
            if not name:
                continue
            ints = []
            for tok in parts:
                if tok != name:
                    try:
                        ints.append(int(float(tok)))
                    except ValueError:
                        pass
            # classification lines carry 2-3 ints (grade, count). Detection box
            # lines carry 4+; a box whose first coordinate lands in 0..3 would
            # otherwise overwrite the label with coordinates.
            if 2 <= len(ints) <= 3 and Path(name).name in imgs:
                g, c = ints[0], ints[1]
                if 0 <= g < N_GRADES and c >= 0:
                    rec[Path(name).name] = {"grade": g, "count": c,
                                            "path": imgs[Path(name).name]}
    if not rec:
        raise SystemExit("no labels parsed - check ROOT")

    counts = [r["count"] for r in rec.values()]
    agree = sum(1 for r in rec.values() if rule(r["count"]) == r["grade"])
    print(f"  images                : {len(rec)}")
    print(f"  lesion counts         : {min(counts)}-{max(counts)}, "
          f"mean {sum(counts)/len(counts):.2f}")
    print(f"  grade == rule(count)  : {agree}/{len(rec)} ({agree/len(rec):.2%})")
    if agree / len(rec) < 0.98:
        raise SystemExit("parser is reading the wrong columns - fix before running")
    return rec


def load_groups(path, names):
    g = {}
    if path is not None:
        with open(path) as fh:
            for r in csv.DictReader(fh):
                g[r["image"]] = r["subject_group"]
    else:
        raise SystemExit("groups.csv not found - attach the fold-audit output")
    for i, n in enumerate(names):
        g.setdefault(n, f"_solo_{i}")
    return g


def load_duplicate_drops(path):
    drop = set()
    if path is None:
        raise SystemExit("duplicate_groups.csv not found - attach verify_duplicates")
    by_group = defaultdict(list)
    with open(path) as fh:
        for r in csv.DictReader(fh):
            by_group[r["group"]].append(r["image"])
    for members in by_group.values():
        for m in sorted(members)[1:]:
            drop.add(m)
    return drop


def kfold(names, group_of, k, seed=SEED):
    """Group-disjoint when group_of separates images; plain random when every
    image is its own group."""
    by_group = defaultdict(list)
    for n in names:
        by_group[group_of[n]].append(n)
    groups = sorted(by_group, key=lambda g: str(g))
    random.Random(seed).shuffle(groups)
    groups.sort(key=lambda g: -len(by_group[g]))
    sizes = [0] * k
    assign = {}
    for g in groups:
        f = min(range(k), key=lambda i: sizes[i])
        assign[g] = f
        sizes[f] += len(by_group[g])
    return {n: assign[group_of[n]] for n in names}


# ---------------------------------------------------------------------------
# metrics
# ---------------------------------------------------------------------------

def qwk(y_true, y_pred, k=N_GRADES):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    O = np.zeros((k, k))
    for t, p in zip(y_true, y_pred):
        O[t, p] += 1
    w = np.array([[((i - j) ** 2) / ((k - 1) ** 2) for j in range(k)]
                  for i in range(k)])
    E = np.outer(np.bincount(y_true, minlength=k),
                 np.bincount(y_pred, minlength=k)) / max(1, len(y_true))
    den = (w * E).sum()
    return 1.0 - (w * O).sum() / den if den > 0 else 0.0


def macro_f1(y_true, y_pred, k=N_GRADES):
    fs = []
    for c in range(k):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        fs.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return float(np.mean(fs))


def bootstrap_ci(fn, *arrays, groups=None, n_boot=500, seed=SEED):
    rng = np.random.default_rng(seed)
    arrays = [np.asarray(a) for a in arrays]
    groups = np.arange(len(arrays[0])) if groups is None else np.asarray(groups)
    uniq = np.unique(groups)
    idx_by = {g: np.where(groups == g)[0] for g in uniq}
    vals = []
    for _ in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by[g] for g in pick])
        try:
            v = fn(*[a[idx] for a in arrays])
            if not (isinstance(v, float) and math.isnan(v)):
                vals.append(v)
        except Exception:
            pass
    if not vals:
        return (float("nan"), float("nan"))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


# ---------------------------------------------------------------------------
# models: a grade head and a count head on the same backbone
# ---------------------------------------------------------------------------

ALLOW_RANDOM_INIT = False   # see the message below before setting this True


def _weights(fn, w):
    """
    Fetch pretrained weights, and explain clearly if the network is off.

    Kaggle notebooks default Internet to OFF, so the torchvision download
    fails with a bare DNS error that says nothing about the real cause.
    """
    import urllib.error, socket
    try:
        return fn(weights=w)
    except (urllib.error.URLError, socket.gaierror, OSError) as e:
        print("\n" + "!" * 70)
        print("  Could not download the pretrained backbone weights.")
        print(f"  ({type(e).__name__}: {e})")
        print()
        print("  Kaggle notebooks start with Internet turned OFF.")
        print("  Fix: right sidebar -> Settings -> Internet -> On, then Run All.")
        print()
        print("  Training from random initialisation instead would change every")
        print("  number and make this run non-comparable with the others, so it")
        print("  is refused by default. Set ALLOW_RANDOM_INIT = True only if you")
        print("  accept that, and label those results as not pretrained.")
        print("!" * 70)
        if not ALLOW_RANDOM_INIT:
            raise SystemExit("stopping: enable Internet in notebook settings")
        print("  ALLOW_RANDOM_INIT is True - continuing WITHOUT pretrained weights.")
        return fn(weights=None)


def build_backbone(name):
    import torch, torchvision
    if name == "resnet18":
        m = _weights(torchvision.models.resnet18,
            torchvision.models.ResNet18_Weights.DEFAULT)
        feat = m.fc.in_features
        m.fc = torch.nn.Identity()
    elif name == "resnet50":
        m = _weights(torchvision.models.resnet50,
            torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        feat = m.fc.in_features
        m.fc = torch.nn.Identity()
    else:
        m = _weights(torchvision.models.efficientnet_b0,
            torchvision.models.EfficientNet_B0_Weights.DEFAULT)
        feat = m.classifier[1].in_features
        m.classifier = torch.nn.Identity()
    return m, feat


def build_model(head, priors=None, mean_count=None):
    """
    head="grade": CORAL ordinal head over grades.
    head="count": scalar regression of the lesion count.

    Both heads carry an informed bias initialisation. Zero-initialised
    thresholds collapse the CORAL head onto the two end classes, and a
    zero-initialised count head starts 13 lesions below every target.
    """
    import torch
    bb, feat = build_backbone(BACKBONE)

    class GradeHead(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = bb
            self.fc = torch.nn.Linear(feat, 1, bias=False)
            q = torch.tensor(priors, dtype=torch.float32).clamp(1e-3, 1 - 1e-3)
            self.bias = torch.nn.Parameter(torch.log(q / (1 - q)))

        def forward(self, x):
            return self.fc(self.backbone(x)) + self.bias

    class CountHead(torch.nn.Module):
        """
        Poisson head: the network emits log(lambda), not the count itself.

        A plain regression with SmoothL1 on counts spanning 1-65 with mean 13
        regresses to the mean and essentially never predicts above 20, so every
        image lands in grade 0 or 1 and the arm collapses - which is exactly
        what the first run did. The log link lets large counts be reached, the
        Poisson likelihood scales its variance with the mean as count data
        does, and - the real payoff - lambda defines a full distribution over
        counts, so grade probabilities follow by summing it over each cut
        interval instead of thresholding a single point estimate.
        """
        def __init__(self):
            super().__init__()
            self.backbone = bb
            self.fc = torch.nn.Linear(feat, 1)
            torch.nn.init.zeros_(self.fc.weight)
            with torch.no_grad():
                self.fc.bias.fill_(float(np.log(max(1e-3, mean_count))))

        def forward(self, x):
            return self.fc(self.backbone(x)).squeeze(1).clamp(-6.0, 6.0)

    class SoftmaxHead(torch.nn.Module):
        """
        Plain 4-way classifier: one logit per grade, no shared scalar.

        CORAL ties every grade to a single scalar plus K-1 thresholds, so the
        grades must line up on one axis and a middle class can be squeezed into
        a narrow band - which is what happened to grade 2 (band width 1.31 in
        s-space against 2.91 for grade 1, predicted 45% as often as it occurs).
        This head gives each grade its own direction, at the cost of not
        enforcing the ordering. It is the diagnostic for whether the grade-2
        failure is the ordinal constraint or the data.
        """
        def __init__(self):
            super().__init__()
            self.backbone = bb
            self.fc = torch.nn.Linear(feat, N_GRADES)
            if priors is not None:
                # start the biases at the log class frequencies so the head
                # does not have to learn the marginal from zero
                q = [1.0 - priors[0]]
                for i in range(len(priors) - 1):
                    q.append(priors[i] - priors[i + 1])
                q.append(priors[-1])
                q = torch.tensor(q, dtype=torch.float32).clamp(1e-3, 1.0)
                with torch.no_grad():
                    self.fc.bias.copy_(torch.log(q / q.sum()))

        def forward(self, x):
            return self.fc(self.backbone(x))

    if head == "grade":
        return GradeHead()
    if head == "softmax":
        return SoftmaxHead()
    return CountHead()


def grade_from_logits(logits):
    """CORAL threshold rule with monotonicity repair. Never argmax: the
    differenced distribution's middle entries are structurally small."""
    p = 1.0 / (1.0 + np.exp(-logits))
    p = np.minimum.accumulate(p, axis=1)
    return (p > 0.5).sum(axis=1).astype(int)


def grade_probs_from_lambda(lam, cuts=CUTS):
    """
    P(grade = g) = P(count in interval_g) under Poisson(lambda).

    This is the honest way to get a grade from a count model: the dataset
    defines grade as an interval of counts, so integrate the predicted count
    distribution over exactly those intervals.
    """
    from scipy.stats import poisson
    lam = np.clip(np.asarray(lam, dtype=float), 1e-6, None)
    edges = list(cuts)
    cdfs = [poisson.cdf(e, lam) for e in edges]
    cols = [cdfs[0]]
    for i in range(1, len(edges)):
        cols.append(cdfs[i] - cdfs[i - 1])
    cols.append(1.0 - cdfs[-1])
    P = np.column_stack(cols)
    return P / np.clip(P.sum(axis=1, keepdims=True), 1e-12, None)


def grade_from_lambda(lam, cuts=CUTS):
    return grade_probs_from_lambda(lam, cuts).argmax(axis=1)


def grade_from_counts(counts, cuts=CUTS):
    c = np.clip(np.rint(np.asarray(counts, dtype=float)), 0, None)
    return np.array([rule(int(v), cuts) for v in c], dtype=int)


# ---------------------------------------------------------------------------
# one fold
# ---------------------------------------------------------------------------

def run_fold(rec, names, fold_of, groups_of, fold, head, device, log):
    import torch
    from torch.utils.data import Dataset, DataLoader
    import torchvision.transforms as T
    from PIL import Image

    test = [n for n in names if fold_of[n] == fold]
    pool = [n for n in names if fold_of[n] != fold]

    pool_groups = sorted({groups_of[n] for n in pool})
    rng = random.Random(SEED + fold)
    rng.shuffle(pool_groups)
    vg = set(pool_groups[:max(1, int(len(pool_groups) * VAL_FRAC))])
    val = [n for n in pool if groups_of[n] in vg]
    train = [n for n in pool if groups_of[n] not in vg]

    tf_tr = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(), T.ColorJitter(0.2, 0.2, 0.2, 0.05),
        T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    tf_ev = T.Compose([
        T.Resize(int(IMG_SIZE * 1.14)), T.CenterCrop(IMG_SIZE), T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    class DS(Dataset):
        def __init__(self, items, tf):
            self.items, self.tf = items, tf
        def __len__(self):
            return len(self.items)
        def __getitem__(self, i):
            n = self.items[i]
            img = Image.open(rec[n]["path"]).convert("RGB")
            return (self.tf(img), rec[n]["grade"], float(rec[n]["count"]))

    dl_tr = DataLoader(DS(train, tf_tr), batch_size=BATCH, shuffle=True,
                       num_workers=2)
    dl_va = DataLoader(DS(val, tf_ev), batch_size=BATCH, num_workers=2)
    dl_te = DataLoader(DS(test, tf_ev), batch_size=BATCH, num_workers=2)

    tr_y = [rec[n]["grade"] for n in train]
    priors = [sum(1 for v in tr_y if v > kk) / max(1, len(tr_y))
              for kk in range(N_GRADES - 1)]
    mean_count = float(np.mean([rec[n]["count"] for n in train]))

    model = build_model(head, priors, mean_count).to(device)
    nodecay = [p for nm, p in model.named_parameters() if nm.endswith("bias")]
    decay = [p for nm, p in model.named_parameters() if not nm.endswith("bias")]
    opt = torch.optim.AdamW(
        [{"params": decay, "weight_decay": WEIGHT_DECAY},
         {"params": nodecay, "weight_decay": 0.0,
          "lr": LR * (10 if head == "grade" else 1)}], lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    bce = torch.nn.BCEWithLogitsLoss()
    if head == "softmax" and CLASS_WEIGHTS:
        cnt = np.bincount(np.array(tr_y), minlength=N_GRADES).astype(float)
        w = torch.tensor((cnt.sum() / np.clip(cnt, 1, None)) / N_GRADES,
                         dtype=torch.float32, device=device)
        ce = torch.nn.CrossEntropyLoss(weight=w)
    else:
        ce = torch.nn.CrossEntropyLoss()

    def fwd_loss(x, y, c):
        out = model(x)
        if head == "grade":
            tgt = (y.unsqueeze(1) >
                   torch.arange(N_GRADES - 1, device=y.device).unsqueeze(0)).float()
            return out, bce(out, tgt)
        if head == "softmax":
            return out, ce(out, y)
        # Poisson NLL up to a constant in c: lambda - c*log(lambda)
        lam = torch.exp(out)
        return out, (lam - c * out).mean()

    def evaluate(dl):
        model.eval()
        P, Y, C = [], [], []
        with torch.no_grad():
            for x, y, c in dl:
                o = model(x.to(device)).cpu().numpy()
                P.append(o); Y.append(y.numpy()); C.append(c.numpy())
        return np.concatenate(P), np.concatenate(Y), np.concatenate(C)

    best, best_state, best_ep = -2.0, None, -1
    for ep in range(EPOCHS):
        model.train()
        for x, y, c in dl_tr:
            x, y, c = x.to(device), y.to(device), c.to(device).float()
            opt.zero_grad()
            _, loss = fwd_loss(x, y, c)
            loss.backward()
            opt.step()
        sched.step()
        vp, vy, _ = evaluate(dl_va)
        vg_ = (grade_from_logits(vp) if head == "grade"
               else vp.argmax(axis=1) if head == "softmax"
               else grade_from_lambda(np.exp(vp)))
        v = qwk(vy, vg_)
        if v > best:
            best, best_ep = v, ep
            best_state = {k: t.detach().cpu().clone()
                          for k, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f"      fold {fold}: train {len(train)} val {len(val)} test {len(test)}"
          f"  best ep {best_ep} val qwk {best:.4f}")

    tp, ty, tc = evaluate(dl_te)
    pg = (grade_from_logits(tp) if head == "grade"
          else tp.argmax(axis=1) if head == "softmax"
          else grade_from_lambda(np.exp(tp)))
    for i, n in enumerate(test):
        log.append({"image": n, "fold": fold, "subject_group": groups_of[n],
                    "true_grade": int(ty[i]), "true_count": float(tc[i]),
                    "pred_grade": int(pg[i]),
                    "pred_count": (float(np.exp(tp[i])) if head == "count" else ""),
                    "true_grade_n": int(ty[i]),
                    "head": head})
    return pg, ty


# ---------------------------------------------------------------------------
# arms and reporting
# ---------------------------------------------------------------------------

def run_arm(name, rec_all, drops, groups_all, use_groups, drop_dups, head,
            device, out):
    print("\n" + "=" * 74)
    print(f"ARM {name}   split={'subject-disjoint' if use_groups else 'random'}"
          f"   duplicates={'removed' if drop_dups else 'kept'}   head={head}")
    print("=" * 74)

    rec = {k: v for k, v in rec_all.items() if not (drop_dups and k in drops)}
    names = sorted(rec)
    groups_of = ({n: groups_all[n] for n in names} if use_groups
                 else {n: f"_solo_{i}" for i, n in enumerate(names)})
    print(f"    images {len(names)}   groups {len({groups_of[n] for n in names})}")

    fold_of = kfold(names, groups_of, N_FOLDS)
    log = []
    for f in range(N_FOLDS):
        run_fold(rec, names, fold_of, groups_of, f, head, device, log)

    yt = np.array([r["true_grade"] for r in log])
    yp = np.array([r["pred_grade"] for r in log])
    gr = np.array([r["subject_group"] for r in log])
    seen = sorted(set(yp.tolist()))
    if len(seen) < len(set(yt.tolist())):
        print(f"    *** WARNING: predicts only {seen}; head collapsed. "
              f"Results for this arm are not interpretable.")

    k = qwk(yt, yp)
    lo, hi = bootstrap_ci(qwk, yt, yp, groups=gr)
    per_grade = {}
    for g in range(N_GRADES):
        m = yt == g
        if m.any():
            per_grade[g] = {"n": int(m.sum()),
                            "predicted": int((yp == g).sum()),
                            "error_rate": float((yp[m] != g).mean())}
    res_extra = {"per_grade": per_grade}

    res = {"arm": name, "n": len(log), "qwk": k, "qwk_lo": lo, "qwk_hi": hi,
           "acc": float((yt == yp).mean()), "macro_f1": macro_f1(yt, yp),
           "mae": float(np.abs(yt - yp).mean()),
           "pred_dist": dict(sorted(Counter(yp.tolist()).items())),
           "head": head, "use_groups": use_groups, "drop_dups": drop_dups,
           **res_extra}
    print(f"    QWK {k:.4f} [{lo:.4f}, {hi:.4f}]  acc {res['acc']:.4f}  "
          f"macro-F1 {res['macro_f1']:.4f}  MAE {res['mae']:.4f}")
    print(f"    predicted grades: {res['pred_dist']}")
    print(f"    {'grade':>7} {'n':>5} {'predicted':>10} {'error rate':>11}")
    for g, pg in sorted(per_grade.items()):
        flag = "  <-- squeezed" if pg["error_rate"] > 0.5 else ""
        print(f"    {g:>7} {pg['n']:>5} {pg['predicted']:>10} "
              f"{pg['error_rate']:>10.1%}{flag}")

    if head == "count":
        pc = np.array([float(r["pred_count"]) for r in log])
        tc = np.array([float(r["true_count"]) for r in log])
        res["count_mae"] = float(np.abs(pc - tc).mean())
        res["count_r"] = float(np.corrcoef(pc, tc)[0, 1])
        res["pred_count_range"] = [float(pc.min()), float(pc.max())]
        print(f"    predicted counts: {pc.min():.1f}-{pc.max():.1f} "
              f"(true {tc.min():.0f}-{tc.max():.0f})   "
              f"MAE {res['count_mae']:.2f}   r {res['count_r']:.3f}")

    with (out / f"log_arm_{name}.csv").open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(log[0]))
        w.writeheader()
        w.writerows(log)
    return res, log


def threshold_sensitivity(log, out):
    """
    Free consequence experiment: re-threshold the SAME predicted counts under
    perturbed cut-points. No retraining, so any change is attributable purely
    to where the thresholds sit.
    """
    pc = np.array([float(r["pred_count"]) for r in log])
    tc = np.array([float(r["true_count"]) for r in log])

    print("\n" + "=" * 74)
    print("THRESHOLD SENSITIVITY  (same model, same predictions, moved cuts)")
    print("=" * 74)
    print("  Both true and predicted grades are re-derived under each cut set,")
    print("  exactly as the dataset itself derives grades from counts.")
    print(f"\n  {'cut-points':>18} {'accuracy':>10} {'QWK':>8} {'delta acc':>11}")

    variants = [CUTS, (4, 20, 50), (6, 20, 50), (5, 19, 50), (5, 21, 50),
                (5, 20, 42), (5, 20, 45)]
    rows, base = [], None
    for cuts in variants:
        # true grades come from the TRUE counts, exactly as the dataset derives
        # them; predicted grades integrate the predicted Poisson over the same
        # intervals, matching the rule used at test time.
        yt = grade_from_counts(tc, cuts)
        yp = grade_from_lambda(pc, cuts)
        acc = float((yt == yp).mean())
        k = qwk(yt, yp)
        if base is None:
            base = acc
        tag = "  <- published" if cuts == CUTS else ""
        print(f"  {str(cuts):>18} {acc:>10.4f} {k:>8.4f} "
              f"{acc-base:>+11.4f}{tag}")
        rows.append({"cuts": str(cuts), "accuracy": acc, "qwk": k,
                     "delta_acc": acc - base})

    spread = max(r["accuracy"] for r in rows) - min(r["accuracy"] for r in rows)
    print(f"\n  accuracy spread across cut sets: {spread:.4f} "
          f"({spread*100:.1f} percentage points)")
    if spread >= 0.02:
        print("  Reported grading accuracy moves materially when an arbitrary")
        print("  threshold shifts by one lesion. It is partly a measurement of")
        print("  threshold placement, not of severity understanding.")
    else:
        print("  Accuracy is stable under small threshold changes. Report this")
        print("  as a negative result; do not claim threshold sensitivity.")

    (out / "threshold_sensitivity.json").write_text(json.dumps(rows, indent=2))
    return rows


def main():
    import torch
    root, out = Path(ROOT), Path(OUT)
    out.mkdir(parents=True, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    print("=" * 74)
    print("CONSEQUENCE EXPERIMENTS")
    print("=" * 74)
    rec_all = load_records(root)
    drops = load_duplicate_drops(find_file("duplicate_groups.csv", DUPLICATES_CSV))
    groups_all = load_groups(find_file("groups.csv", GROUPS_CSV), sorted(rec_all))
    print(f"  duplicate files to drop: {len(drops)}")

    if USE_IMAGE_CACHE:
        rec_all = build_image_cache(rec_all)

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  device                 : {dev}")
    if dev == "cpu":
        print("  WARNING: no GPU - this will take many hours. Enable the accelerator.")

    spec = {"A": (False, False, "grade"),
            "B": (True,  True,  "grade"),
            "C": (True,  True,  "count"),
            "D": (True,  True,  "softmax")}
    print(f"  arms to run            : {ARMS}"
          + (f"   (reusing stored results for {sorted(set(spec) - set(ARMS))})"
             if set(ARMS) != set(spec) else ""))

    got, logC = {}, None
    for a in ARMS:
        ug, dd, hd = spec[a]
        r, lg = run_arm(a, rec_all, drops, groups_all, ug, dd, hd, dev, out)
        got[a] = r
        if a == "C":
            logC = lg

    def arm(a):
        """This run's result if the arm ran, else the stored reference."""
        if a in got:
            return got[a], True
        if a in REFERENCE:
            r = dict(REFERENCE[a]); r.update(
                {"arm": a, "n": None, "head": spec[a][2],
                 "use_groups": spec[a][0], "drop_dups": spec[a][1]})
            return r, False
        return None, False

    rA, ranA = arm("A")
    rB, ranB = arm("B")
    rC, ranC = arm("C")
    rD, ranD = arm("D")
    results = [r for r in (rA, rB, rC, rD) if r is not None]

    sens = threshold_sensitivity(logC, out) if logC else None

    print("\n" + "=" * 74)
    print("SUMMARY")
    print("=" * 74)
    print(f"  {'arm':>4} {'n':>6} {'split':>18} {'dups':>9} {'head':>7} "
          f"{'QWK':>8} {'acc':>8}")
    for r in results:
        tag = "" if r.get("n") else "   (stored)"
        print(f"  {r['arm']:>4} {str(r.get('n') or '-'):>6} "
              f"{('subject' if r['use_groups'] else 'random'):>18} "
              f"{('removed' if r['drop_dups'] else 'kept'):>9} {r['head']:>7} "
              f"{r['qwk']:>8.4f} {r['acc']:>8.4f}{tag}")

    print("\n" + "-" * 74)
    print("WHAT THE ARMS SAY")
    print("-" * 74)
    dq = rA["qwk"] - rB["qwk"]
    da = rA["acc"] - rB["acc"]
    print(f"  A - B  (protocol inflation) : QWK {dq:+.4f}   accuracy {da:+.4f}")
    if abs(da) < 0.01 and abs(dq) < 0.02:
        print("    The protocol defects do not materially inflate reported scores.")
        print("    Report this plainly - it is an honest negative result and it")
        print("    strengthens the paper's credibility. The redundancy argument")
        print("    does not depend on it.")
    else:
        print("    Evaluating under the benchmark's own protocol overstates")
        print("    performance by the amounts above.")

    dq2 = rC["qwk"] - rB["qwk"]
    da2 = rC["acc"] - rB["acc"]
    print(f"\n  C - B  (count head vs grade head) : QWK {dq2:+.4f}  "
          f"accuracy {da2:+.4f}")
    if da2 >= -0.02:
        print("    A model that never sees a grade label - it predicts the lesion")
        print("    count and applies three thresholds - matches or beats the")
        print("    dedicated grading model. The grading head contributes nothing")
        print("    the counting head does not already determine. This is the")
        print("    central claim, demonstrated rather than argued.")
    else:
        print("    The grading head outperforms count-plus-thresholds. Since the")
        print("    labels are exactly rule(count), this gap is an optimisation")
        print("    artifact, not extra information: report it as such and do not")
        print("    claim the two are equivalent in practice.")

    if "D" in got and "per_grade" in got["D"]:
        print("\n" + "-" * 74)
        print("DID THE ORDINAL CONSTRAINT CAUSE THE GRADE-2 FAILURE?")
        print("-" * 74)
        d2 = got["D"]["per_grade"].get(2)
        if d2:
            print(f"  CORAL head (arm B, stored)  : grade 2 error 70.5%, "
                  f"predicted 80 of 176")
            print(f"  softmax head (arm D)        : grade 2 error "
                  f"{d2['error_rate']:.1%}, predicted {d2['predicted']} of {d2['n']}")
            if d2["error_rate"] < 0.55:
                print("\n  The ordinal head was the problem, not the data. Report the")
                print("  softmax baseline as the paper's headline accuracy, and re-run")
                print("  the boundary-margin test on THIS model before concluding")
                print("  anything about the margin - the earlier verdict was measured")
                print("  against errors this architecture created.")
            else:
                print("\n  Grade 2 fails under both heads, so it is the data, not the")
                print("  ordinal constraint: 176 images spanning counts 21-50, the")
                print("  second-rarest class, bordered on both sides. The earlier")
                print("  boundary-margin verdict stands.")
    print()
    (out / "summary.json").write_text(json.dumps(
        {"arms": results, "threshold_sensitivity": sens,
         "arms_run_this_session": ARMS,
         "config": {"backbone": BACKBONE, "epochs": EPOCHS, "folds": N_FOLDS,
                    "seed": SEED}}, indent=2, default=float))
    print(f"\n  wrote {out}/summary.json and per-arm logs")
    print("=" * 74)


main()
